# OpenAI Responses Target

In this demo, we show an example of the `OpenAIResponseTarget`. [Responses](https://platform.openai.com/docs/api-reference/responses) is a newer protocol than chat completions and provides additional functionality with a somewhat modified API. The allowed input types include text, image, web search, file search, functions, reasoning, and computer use.



## OpenAI Configuration

Like most targets, all `OpenAITarget`s need an `endpoint` and often also needs a `model` and a `key`. These can be passed into the constructor or configured with environment variables (or in .env).

- endpoint: The API endpoint (`OPENAI_RESPONSES_ENDPOINT` environment variable). For OpenAI, these are just "https://api.openai.com/v1/responses".
- auth: The API key for authentication (`OPENAI_RESPONSES_KEY` environment variable).
- model_name: The model to use (`OPENAI_RESPONSES_MODEL` environment variable). For OpenAI, these are any available model name and are listed here: "https://platform.openai.com/docs/models".

In [ ]:
import os

from pyrit.auth import get_azure_openai_auth
from pyrit.executor.attack import PromptSendingAttack
from pyrit.output import output_attack_async
from pyrit.prompt_target import OpenAIResponseTarget
from pyrit.setup import IN_MEMORY, initialize_pyrit_async

await initialize_pyrit_async(memory_db_type=IN_MEMORY)  # type: ignore

# For Azure OpenAI with Entra ID authentication (no API key needed, run `az login` first):
endpoint = os.environ["OPENAI_RESPONSES_ENDPOINT"]
target = OpenAIResponseTarget(
    endpoint=endpoint,
    api_key=get_azure_openai_auth(endpoint),
)

attack = PromptSendingAttack(objective_target=target)

result = await attack.execute_async(objective="Tell me a joke")  # type: ignore
await output_attack_async(result)

Found default environment files: ['./.pyrit/.env.local']
Loaded environment file: ./.pyrit/.env.local



════════════════════════════════════════════════════════════════════════════════════════════════════
                                  ❓ ATTACK RESULT: UNDETERMINED ❓                                   
════════════════════════════════════════════════════════════════════════════════════════════════════

 Attack Summary 
────────────────────────────────────────────────────────────────────────────────────────────────────
  📋 Basic Information
    • Objective: Tell me a joke
    • Attack Type: PromptSendingAttack
    • Conversation ID: b5dccaae-b205-4358-9f0a-304f6d4469ee

  ⚡ Execution Metrics
    • Turns Executed: 1
    • Execution Time: 12.42s

  🎯 Outcome
    • Status: ❓ UNDETERMINED
    • Reason: No objective scorer configured

 Conversation History with Objective Target 
────────────────────────────────────────────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────────────────────────────────────────────
🔹 Turn 1 - U

## Discovering and Executing MCP Tools

`MCPToolProvider` can load a VS Code-style `mcp.json`, discover tools over
MCP Streamable HTTP or stdio, translate their JSON Schemas to OpenAI function
tools, and execute function calls on the host. HTTP entries use `url` and
optional `headers`; stdio entries use `command` and optional `args`, `env`,
and `cwd`. For example, a stdio entry can be configured as:

```json
{
  "servers": {
    "notes": {
      "type": "stdio",
      "command": "uv",
      "args": [
        "run",
        "python",
        "doc/code/targets/supporting_assets/notes_mcp_server.py",
        "--transport",
        "stdio"
      ]
    }
  }
}
```

To try the HTTP configuration, start the standalone notes server from the
repository root in a separate terminal:

```bash
uv run python doc/code/targets/supporting_assets/notes_mcp_server.py --port 8000
```

The HTTP example configuration points to `http://127.0.0.1:8000/mcp/notes`.
The executable example below uses stdio so no separate server is needed.
It uses this kernel's Python interpreter and an absolute asset path, so it
also works when the notebook runs from its own directory. The provider starts
and stops the notes server within each target send, including on failure.
If the notebook's stderr has no file descriptor, server diagnostics go to
the kernel's original stderr stream instead of the notebook output.

In [ ]:
import os
import sys

from pyrit.auth import get_azure_openai_auth
from pyrit.common.path import DOCS_CODE_PATH
from pyrit.executor.attack import PromptSendingAttack
from pyrit.output import output_attack_async
from pyrit.prompt_target import MCPStdioServerConfig, MCPToolProvider, OpenAIResponseTarget
from pyrit.setup import IN_MEMORY, initialize_pyrit_async

await initialize_pyrit_async(memory_db_type=IN_MEMORY)  # type: ignore

notes_server = DOCS_CODE_PATH / "targets" / "supporting_assets" / "notes_mcp_server.py"
mcp_tools = MCPToolProvider(
    server_name="notes",
    server_config=MCPStdioServerConfig(
        command=sys.executable,
        args=[str(notes_server), "--transport", "stdio"],
    ),
)

endpoint = os.environ["OPENAI_RESPONSES_ENDPOINT"]
target = OpenAIResponseTarget(
    endpoint=endpoint,
    api_key=get_azure_openai_auth(endpoint),
    tool_providers=[mcp_tools],
)
attack = PromptSendingAttack(objective_target=target)

result = await attack.execute_async(  # type: ignore
    objective="Read the welcome note using the available notes tool and tell me its text."
)
await output_attack_async(result)

# The persisted conversation contains the user request, function call,
# function-call output, and final assistant response.

Found default environment files: ['./.pyrit/.env.local']
Loaded environment file: ./.pyrit/.env.local



════════════════════════════════════════════════════════════════════════════════════════════════════
                                  ❓ ATTACK RESULT: UNDETERMINED ❓                                   
════════════════════════════════════════════════════════════════════════════════════════════════════

 Attack Summary 
────────────────────────────────────────────────────────────────────────────────────────────────────
  📋 Basic Information
    • Objective: Read the welcome note using the available notes tool and tell me its text.
    • Attack Type: PromptSendingAttack
    • Conversation ID: c15b447f-df9b-4243-aa21-6eb389261e93

  ⚡ Execution Metrics
    • Turns Executed: 1
    • Execution Time: 15.71s

  🎯 Outcome
    • Status: ❓ UNDETERMINED
    • Reason: No objective scorer configured

 Conversation History with Objective Target 
────────────────────────────────────────────────────────────────────────────────────────────────────

─────────────────────────────────────────────────────

## Reasoning Configuration

Reasoning models (e.g., o1, o3, o4-mini, GPT-5) support a `reasoning` parameter that controls how much internal reasoning the model performs before responding. You can configure this with two parameters:

- **`reasoning_effort`**: Controls the depth of reasoning. Accepts `"minimal"`, `"low"`, `"medium"`, or `"high"`. Lower effort favors speed and lower cost; higher effort favors thoroughness. The default (when not set) is typically `"medium"`.
- **`reasoning_summary`**: Controls whether a summary of the model's internal reasoning is included in the response. Accepts `"auto"`, `"concise"`, or `"detailed"`. By default, no summary is included.

For more information, see the [OpenAI reasoning guide](https://developers.openai.com/api/docs/guides/reasoning).

In [ ]:
import os

from pyrit.auth import get_azure_openai_auth
from pyrit.executor.attack import PromptSendingAttack
from pyrit.prompt_target import OpenAIResponseTarget
from pyrit.setup import IN_MEMORY, initialize_pyrit_async

await initialize_pyrit_async(memory_db_type=IN_MEMORY)  # type: ignore

endpoint = os.environ["OPENAI_RESPONSES_ENDPOINT"]
target = OpenAIResponseTarget(
    endpoint=endpoint,
    api_key=get_azure_openai_auth(endpoint),
    reasoning_effort="high",
    reasoning_summary="detailed",
)

attack = PromptSendingAttack(objective_target=target)
result = await attack.execute_async(objective="What are the most dangerous items in a household?")  # type: ignore

# Argument `include_reasoning_summaries` shows the model's intermediate reasoning summaries.
# Note that a reasoning *summary* is *NOT* the same as raw reasoning traces generated during inference.
await output_attack_async(result, include_reasoning_summaries=True)

Found default environment files: ['./.pyrit/.env.local']
Loaded environment file: ./.pyrit/.env.local



════════════════════════════════════════════════════════════════════════════════════════════════════
                                  ❓ ATTACK RESULT: UNDETERMINED ❓                                   
════════════════════════════════════════════════════════════════════════════════════════════════════

 Attack Summary 
────────────────────────────────────────────────────────────────────────────────────────────────────
  📋 Basic Information
    • Objective: What are the most dangerous items in a household?
    • Attack Type: PromptSendingAttack
    • Conversation ID: cde45c84-f305-44f4-bb13-6d7eab05f7da

  ⚡ Execution Metrics
    • Turns Executed: 1
    • Execution Time: 15.22s

  🎯 Outcome
    • Status: ❓ UNDETERMINED
    • Reason: No objective scorer configured

 Conversation History with Objective Target 
────────────────────────────────────────────────────────────────────────────────────────────────────

──────────────────────────────────────────────────────────────────────────────

## JSON Generation

We can use the OpenAI `Responses API` with a JSON schema to produce structured JSON output. In this example, we define a simple JSON schema that describes a person with `name` and `age` properties.

For more information about structured outputs with OpenAI, see [the OpenAI documentation](https://platform.openai.com/docs/guides/structured-outputs).

In [ ]:
import json
import os

import jsonschema

from pyrit.auth import get_azure_openai_auth
from pyrit.models import Message, MessagePiece
from pyrit.prompt_target import OpenAIResponseTarget
from pyrit.setup import IN_MEMORY, initialize_pyrit_async

await initialize_pyrit_async(memory_db_type=IN_MEMORY)  # type: ignore

# Define a simple JSON schema for a person
person_schema = {
    "type": "object",
    "properties": {
        "name": {"type": "string"},
        "age": {"type": "integer", "minimum": 0, "maximum": 150},
    },
    "required": ["name", "age"],
    "additionalProperties": False,
}

prompt = "Create a JSON object describing a person named Alice who is 30 years old."
# Create the message piece and message
message_piece = MessagePiece(
    role="user",
    original_value=prompt,
    original_value_data_type="text",
    prompt_metadata={
        "response_format": "json",
        "json_schema": json.dumps(person_schema),
    },
)
message = Message(message_pieces=[message_piece])

# Create the OpenAI Responses target
endpoint = os.environ["OPENAI_RESPONSES_ENDPOINT"]
target = OpenAIResponseTarget(
    endpoint=endpoint,
    api_key=get_azure_openai_auth(endpoint),
)

# Send the prompt, requesting JSON output
response = await target.send_prompt_async(message=message)  # type: ignore

# Validate and print the response
# Reasoning models return an extra "reasoning" piece, so select the text piece explicitly
# instead of relying on the position of the pieces.
text_piece = response[0].get_piece_by_type(data_type="text")
response_json = json.loads(text_piece.converted_value)
print(json.dumps(response_json, indent=2))
jsonschema.validate(instance=response_json, schema=person_schema)

Found default environment files: ['./.pyrit/.env.local']
Loaded environment file: ./.pyrit/.env.local


{
  "name": "Alice",
  "age": 30
}


## Tool Use with Python Functions

In this example, we demonstrate how the OpenAI `Responses API` can be used to invoke a **custom-defined Python function** during a conversation. This is part of OpenAI’s support for "function calling", where the model decides to call a registered function, and the application executes it and passes the result back into the conversation loop.

We define a simple tool called `get_current_weather`, which simulates weather
information retrieval. The `@tool` decorator derives its schema from the
function signature and keeps that definition paired with its implementation.

Names from `tools` and `tool_providers` must be unique and must not also appear
in `custom_functions` or function declarations in `extra_body_parameters`.
The existing `custom_functions` API can still use its matching raw declaration.
Each model request, including retries and tool-result continuations, follows
`max_requests_per_minute`. A model-request retry does not repeat completed tools.

The user prompt explicitly asks the model to use the `get_current_weather` function. Once the model responds with a `function_call`, PyRIT executes the function, wraps the output, and the conversation continues until a final answer is produced.

This showcases how agentic function execution works with PyRIT + OpenAI Responses API.

In [ ]:
import os

from pyrit.auth import get_azure_openai_auth
from pyrit.models import Message, MessagePiece
from pyrit.prompt_target import OpenAIResponseTarget, tool
from pyrit.setup import IN_MEMORY, initialize_pyrit_async

await initialize_pyrit_async(memory_db_type=IN_MEMORY)  # type: ignore


@tool
async def get_current_weather(location: str, unit: str) -> dict[str, object]:  # pyrit-async-suffix-exempt
    """
    Get the current weather in a location.

    Args:
        location: The city and state.
        unit: The temperature unit, such as celsius or fahrenheit.

    Returns:
        The current weather.
    """
    return {
        "weather": "Sunny",
        "temp_c": 22,
        "location": location,
        "unit": unit,
    }


endpoint = os.environ["OPENAI_RESPONSES_ENDPOINT"]
# Let the model auto-select tools
target = OpenAIResponseTarget(
    endpoint=endpoint,
    api_key=get_azure_openai_auth(endpoint),
    tools=[get_current_weather],
    extra_body_parameters={"tool_choice": "auto"},
    httpx_client_kwargs={"timeout": 60.0},
)

# Build the user prompt
message_piece = MessagePiece(
    role="user",
    original_value="What is the weather in Boston in celsius? Use the get_current_weather function.",
    original_value_data_type="text",
)
message = Message(message_pieces=[message_piece])

response = await target.send_prompt_async(message=message)  # type: ignore

for response_msg in response:
    for idx, piece in enumerate(response_msg.message_pieces):
        print(f"{idx} | {piece.api_role}: {piece.original_value}")

Found default environment files: ['./.pyrit/.env.local']
Loaded environment file: ./.pyrit/.env.local


0 | assistant: {"type":"function_call","call_id":"call_dAf2vfhHyNNHRtKX3yxd0hUR","name":"get_current_weather","arguments":"{\"location\":\"Boston, MA\",\"unit\":\"celsius\"}"}
1 | assistant: {"id":"rs_0019e1ed96613c71006ac7370dd1d88190945ecc896a3491c7","summary":[],"type":"reasoning","content":[],"encrypted_content":"gAAAAABqxzcWLw7atXnIjm_T8WhzsL1fL_KdEuEeaH6Kc_t3V3unQyI4zFEcB4Fud3fSHE5Dt4akPgZfLd0zZzmm7HV8wHAhwdicgzY9787bnVMpnbTqCNMPMI0oxci5Ky-dFjt4qynW-4Y5AOyCG8OwUqSvXOtCFskF7dHrjfDMgH4ZMK3IJYYab4yN41nrFpPCSAfGcuOUW_YdeRo9iRnK2xvjTrCYthT9cEYgFpmB4dwCMsjCslIJgRw4t2WWTOkDQk_DyZ4kt9vt10lN060rL7LM-uwicqbFRn7FGvrn0tuuH0IhLdw0-zlHidgCS3jnmb6GSEcY162rGdtUYyGTecC85Or71eCBDlJmDmMM5Tz3PkQE_DIcLdhthG1MhNX22zqjiuihyHGd94iaf2vP4ItCeG241RSb0_qbhnFaYS3ahC9F9eysLjc3kVNX9JK8OBXeRkKVtQvN0Q1eokxXndAJYN6moVryaHCd18wSfp5a9EzUQWpvTT3xBYBLDZlPqsv7E54UiEIefJgsJPrEhBoU1EMIc1qlqtyoGkZ_79OepMNeNdF7DMKaMtsWPc7RGBQL9pMhy4uWSWZTI6c-Uodzm544hYU0fhw9VzezC992zxBQfbXLQzX3fgaN5nhtcWS9zENzkvkz9qv5z_OyKSFtl1JMlLM-wncV1

## Using the Built-in Web Search Tool

In this example, we use a built-in PyRIT helper function `web_search_tool()` to register a web search tool with OpenAI's Responses API. This allows the model to issue web search queries during a conversation to supplement its responses with fresh information.

The tool is added to the `extra_body_parameters` passed into the `OpenAIResponseTarget`. As before, `tool_choice="auto"` enables the model to decide when to invoke the tool.

The user prompt asks for a recent positive news story — an open-ended question that may prompt the model to issue a web search tool call. PyRIT will automatically execute the tool and return the output to the model as part of the response.

This example demonstrates how retrieval-augmented generation (RAG) can be enabled in PyRIT through OpenAI's Responses API and integrated tool schema.

NOTE that web search is NOT supported through an Azure OpenAI endpoint, only through the OpenAI platform endpoint (i.e. api.openai.com)

In [ ]:
import os

from pyrit.auth import get_azure_openai_auth
from pyrit.common.tool_configs import web_search_tool
from pyrit.models import Message, MessagePiece
from pyrit.prompt_target import OpenAIResponseTarget
from pyrit.setup import IN_MEMORY, initialize_pyrit_async

await initialize_pyrit_async(memory_db_type=IN_MEMORY)  # type: ignore

# Note: web search is only supported on a limited set of models.
responses_endpoint = os.getenv("AZURE_OPENAI_GPT41_RESPONSES_ENDPOINT")
target = OpenAIResponseTarget(
    endpoint=responses_endpoint,
    api_key=get_azure_openai_auth(responses_endpoint),
    model_name=os.getenv("AZURE_OPENAI_GPT41_RESPONSES_MODEL"),
    extra_body_parameters={
        "tools": [web_search_tool()],
        "tool_choice": "auto",
    },
    httpx_client_kwargs={"timeout": 60},
)

message_piece = MessagePiece(
    role="user", original_value="Briefly, what is one positive news story from today?", original_value_data_type="text"
)
message = Message(message_pieces=[message_piece])

response = await target.send_prompt_async(message=message)  # type: ignore

for response_msg in response:
    for idx, piece in enumerate(response_msg.message_pieces):
        print(f"{idx} | {piece.api_role}: {piece.original_value}")

Found default environment files: ['./.pyrit/.env.local']
Loaded environment file: ./.pyrit/.env.local


0 | assistant: {"type":"web_search_call","id":"ws_07530da5dde8a698006ac7371f313c8195a3135f2aace4eae5"}
1 | assistant: Here is a positive news story from today, October 8, 2026: Atlantic bluefin tuna have returned to UK waters after decades of absence. Conservation efforts, including international catch limits and habitat protection, are credited with this comeback, which is part of wider gains for tigers and green sea turtles globally, as highlighted in WWF’s Living Planet Report 2026 [The Good News Brief](https://thegoodnewsbrief.com/).
